# 01 (alt) — Download Data from MAST

Trial notebook exploring obs_id-based download approach.
All parameters read from `config/wfc3_ir_drizzle_params.yaml` — nothing hardcoded here.

**Outputs:** FLT files in `data/raw/<target_name>/`

In [ ]:
import os
import yaml
import glob
import shutil
import warnings
import numpy as np
import matplotlib.pyplot as plt
import drizzlepac
from drizzlepac import tweakreg
from collections import defaultdict
from pathlib import Path
from astropy import wcs
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import FITSFixedWarning
from astropy.visualization import ImageNormalize, LogStretch
from astroquery.mast import Observations
from stwcs.wcsutil import headerlet
from IPython.display import clear_output

# Suppress noisy but harmless warning triggered by HDRLET extensions,
# which carry WCS info but no image data (0 axes vs 2 in the WCS)
warnings.filterwarnings('ignore', category=FITSFixedWarning)

# Load the 'download' section from config — all query parameters live there
cfg = yaml.safe_load(open('../../config/wfc3_ir_drizzle_params.yaml'))['download']

# Paths for raw and processed data
RAW_DIR  = Path('../../data/raw')
PROC_DIR = Path('../../data/processed')

# Query MAST for all observations from our proposal
# project='HST' is required — 'HAP' returns processed mosaics, not individual exposures
obs_table = Observations.query_criteria(
    proposal_id=cfg['proposal_id'],
    instrument_name=cfg['instrument_name'],
    project=cfg['project']
)

print(f"Found {len(obs_table)} observations for proposal {cfg['proposal_id']}")

In [ ]:
# Get the full list of data products linked to those observations
products = Observations.get_product_list(obs_table)

# Download products, applying FLT and CALWF3 filters directly in this call
# This avoids a separate filter_products step — download_products handles it internally
# cache=True skips any files already present on disk
Observations.download_products(
    products,
    productSubGroupDescription=cfg['product_sub_group'],  # 'FLT'
    project=cfg['data_type'],                              # 'CALWF3'
    cache=True
)

In [ ]:
# Build a summary table of key FITS header values for every downloaded FLT file
# Run this before organizing files — lets you verify targets, filters, dither pattern,
# and roll angle before committing to the move step

# fl? matches both flt.fits and flc.fits; path reflects MAST's staging location
flt_files = sorted(glob.glob('./mastDownload/HST/*/*fl?.fits'))

# Keywords to pull from the primary header (ext=0)
keywords_ext0 = ["ROOTNAME", "ASN_ID", "TARGNAME", "DETECTOR", "FILTER", "EXPTIME",
                 "RA_TARG", "DEC_TARG", "POSTARG1", "POSTARG2", "DATE-OBS"]

# ORIENTAT (roll angle) lives in the science extension (ext=1), not the primary header
keywords_ext1 = ["ORIENTAT"]

data = []
for flt_file in flt_files:
    row = []
    # Extract each primary header keyword
    for keyword in keywords_ext0:
        row.append(fits.getval(flt_file, keyword, ext=0))
    # Extract science extension keywords
    for keyword in keywords_ext1:
        row.append(fits.getval(flt_file, keyword, ext=1))
    data.append(row)

# Assemble into an astropy Table with appropriate column types
keywords = keywords_ext0 + keywords_ext1
table = Table(
    np.array(data),
    names=keywords,
    dtype=['str', 'str', 'str', 'str', 'str', 'f8', 'f8', 'f8', 'f8', 'f8', 'str', 'f8']
)

# Format numeric columns for readability
table['EXPTIME'].format = '7.1f'
table['RA_TARG'].format = table['DEC_TARG'].format = '7.4f'
table['POSTARG1'].format = table['POSTARG2'].format = '7.3f'
table['ORIENTAT'].format = '7.2f'

print(f"{len(flt_files)} FLT files found")
table

In [ ]:
# MAST downloads files into ./mastDownload/HST/<visit>/<filename>
# This cell moves each file into data/raw/<TARGNAME>/ using the FITS header
# TARGNAME from the header is more reliable than MAST metadata for folder naming
for flt in glob.glob('./mastDownload/HST/*/*flt.fits'):
    flt_path = Path(flt)

    # Read the target name directly from the FITS primary header
    target = fits.getval(str(flt_path), 'TARGNAME', ext=0)

    # Create the per-target folder if it doesn't already exist
    dest_dir = RAW_DIR / target
    dest_dir.mkdir(parents=True, exist_ok=True)

    # Move the file from the MAST staging folder into its target folder
    dest = dest_dir / flt_path.name
    shutil.move(str(flt_path), str(dest))
    print(f"Moved {flt_path.name} → {dest}")

# Uncomment after confirming all files moved correctly:
# shutil.rmtree('./mastDownload/')

In [ ]:
# idcscale is an instrument property — the same for all WFC3/IR files, so read it once
plate_scale = fits.getval(str(sorted(RAW_DIR.glob('*/*flt.fits'))[0]), 'idcscale', ext=1)
print(f'Detector plate scale: {plate_scale:.4f} arcsec/pixel')

def plot_dither(flt_list, axes, plate_scale):
    # Build arrays of POSTARG offsets and their sub-pixel phases for a list of FLT files
    n = len(flt_list)
    postarg1 = np.empty(n, dtype=np.float32)
    postarg2 = np.empty(n, dtype=np.float32)
    x_phase  = np.empty(n, dtype=np.float32)
    y_phase  = np.empty(n, dtype=np.float32)

    for i, flt in enumerate(flt_list):
        with fits.open(str(flt)) as hdu:
            postarg1[i] = hdu[0].header['POSTARG1']
            postarg2[i] = hdu[0].header['POSTARG2']
            # modf splits a value into (fractional, integer) parts
            # dividing by plate_scale converts arcsec offset to pixels;
            # the fractional part is how far each dither falls from a whole pixel
            x_phase[i] = abs(np.modf(postarg1[i] / plate_scale)[0])
            y_phase[i] = abs(np.modf(postarg2[i] / plate_scale)[0])

    # Left panel: sky-plane dither positions in arcsec
    axes[0].plot(postarg1, postarg2, 'k+-', ms=15, lw=1)
    axes[0].set_xlabel('POSTARG1 RA [arcsec]', fontsize='large')
    axes[0].set_ylabel('POSTARG2 DEC [arcsec]', fontsize='large')

    # Right panel: sub-pixel phase — ideally points spread across the unit square
    axes[1].plot(x_phase, y_phase, 'k+-', ms=15, lw=1)
    axes[1].set_xlim(-0.05, 1)
    axes[1].set_ylim(-0.05, 1)
    axes[1].set_xlabel('X pixel phase', fontsize='large')
    axes[1].set_ylabel('Y pixel phase', fontsize='large')

# One combined figure per quasar — both filters together
for quasar_dir in sorted(RAW_DIR.iterdir()):
    flt_files = sorted(quasar_dir.glob('*flt.fits'))
    if not flt_files:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(quasar_dir.name, fontsize='large')
    plot_dither(flt_files, axes, plate_scale)
    plt.tight_layout()
    plt.show()

In [ ]:
# Keywords to read from the primary header (ext=0)
ext_0_kws = ['DETECTOR', 'EXPTIME']

# WCS solution keywords from the science extension (ext=1)
# WCSNAME: name of the active WCS solution applied by the pipeline
# NMATCHES: number of sources matched to the reference catalog
# RMS_RA / RMS_DEC: astrometric residuals in milliarcseconds
ext_1_kws = ['WCSNAME', 'NMATCHES', 'RMS_RA', 'RMS_DEC']

# Physical plate scales per detector in arcsec/pixel — instrument constants
det_scale = {'IR': 0.1283, 'UVIS': 0.0396, 'WFC': 0.05}

# Loop per quasar and print one WCS summary table each
for quasar_dir in sorted(RAW_DIR.iterdir()):
    flt_files = sorted(quasar_dir.glob('*flt.fits'))
    if not flt_files:
        continue

    col_dict = defaultdict(list)

    for f in flt_files:
        # Store just the filename, not the full path, for readability
        col_dict['filename'].append(f.name)
        hdr0 = fits.getheader(str(f), 0)
        hdr1 = fits.getheader(str(f), 1)

        # Read primary header keywords
        for kw in ext_0_kws:
            col_dict[kw].append(hdr0[kw])

        # Read WCS keywords — use .get() so missing keywords show as None rather than crashing
        for kw in ext_1_kws:
            val = hdr1.get(kw, None)
            if val is not None and 'RMS' in kw:
                val = np.around(val, 1)
            col_dict[kw].append(val)

        # Convert RMS to pixels only if the values are present in the header
        for kw in ['RMS_RA', 'RMS_DEC']:
            rms = hdr1.get(kw, None)
            pix_val = np.round(rms / 1000. / det_scale[hdr0['DETECTOR']], 2) if rms is not None else None
            col_dict[f'{kw}_pix'].append(pix_val)

    print(f'\n{quasar_dir.name}')
    display(Table(col_dict))

### ⚠ W2M1042+1641 — Incomplete Pipeline Astrometry

Of the 16 FLT files for this target, only **4 have pipeline astrometric solutions** (NMATCHES / RMS_RA / RMS_DEC populated). The remaining 12 have only the basic IDC instrument distortion WCS — no Gaia-based refinement was applied by the pipeline.

The 4 files with solutions all share identical values, indicating they come from a single visit/association where catalog matching succeeded. The other 3 visits had insufficient catalog coverage or the pipeline skipped astrometric refinement.

**Action required in notebook 03:**
- Run TweakReg on W2M1042+1641 with a Gaia reference catalog to anchor all 16 exposures to a consistent absolute frame.
- Check TweakReg residuals for this target more carefully than the others.
- Do not rely on relative-only alignment here — the 12 IDC-only files may have larger initial offsets.

### WCS Solution Summary — All Quasars

| Quasar | WCSNAME | Catalog | Notes |
|--------|---------|---------|-------|
| W2M0811+0115 | `FIT_REL_GAIAeDR3` | Gaia eDR3 | Best available — direct Gaia eDR3 alignment |
| W2M1042+1641 (4 files) | `FIT_REL_GSC242` | GSC v2.4.2 | One visit succeeded; aligned via drizzled combined image |
| W2M1042+1641 (12 files) | `GSC240` | GSC 2.4.0 × Gaia DR1 | No astrometric refinement beyond IDC baseline |
| W2M1220+1126 | `FIT_REL_GSC242` | GSC v2.4.2 | Aligned as a set via drizzled combined image |
| W2M1252+0715 | `FIT_REL_GSC242` | GSC v2.4.2 | Aligned as a set via drizzled combined image |
| W2M1439+0858 | `FIT_REL_GSC242` | GSC v2.4.2 | Aligned as a set via drizzled combined image |
| W2M1542+1259 | `FIT_REL_GSC242` | GSC v2.4.2 | Aligned as a set via drizzled combined image |

**Next step:** Code reads WCSNAME from each file's header dynamically — no hardcoding — then solves for each image using its unique WCS solution.

In [ ]:
# Reset each file to its pipeline a priori WCS solution using headerlets
# The a priori solution preserves relative alignment between exposures,
# so TweakReg is not needed before drizzling for files that have one.
#
# Process per file:
# 1. Read the active WCSNAME from ext=1 (set by the pipeline)
# 2. Build the list of all WCSNAME values stored in the file's HDRLET extensions
# 3. Find the index of the active WCSNAME in that list
# 4. Add 1 to convert from 0-indexed list to 1-indexed EXTVER
# 5. Restore that headerlet as the active WCS

for quasar_dir in sorted(RAW_DIR.iterdir()):
    flt_files = sorted(quasar_dir.glob('*flt.fits'))
    if not flt_files:
        continue

    print(f'\n{quasar_dir.name}')
    for flt in flt_files:

        # Read the WCSNAME the pipeline assigned to this file
        target_wcsname = fits.getval(str(flt), 'WCSNAME', ext=1)

        # Collect all WCSNAME values from HDRLET extensions in order
        # HDRLET EXTVER is 1-indexed; position in this list is 0-indexed
        with fits.open(str(flt)) as hdu:
            wcsnames = [ext.header.get('WCSNAME', '') for ext in hdu if ext.name == 'HDRLET']

        if target_wcsname in wcsnames:
            # Index + 1 maps the 0-indexed list position to the 1-indexed HDRLET EXTVER
            chosen_ext = wcsnames.index(target_wcsname) + 1
            headerlet.restore_from_headerlet(
                str(flt),
                hdrext=('HDRLET', chosen_ext),
                archive=False,  # do not save current WCS as a new headerlet before restoring
                force=False      # do not overwrite if a conflict exists
            )
            print(f'  {flt.name}: {fits.getval(str(flt), "WCSNAME", ext=1)}')
        else:
            print(f'  {flt.name}: WARNING — "{target_wcsname}" not found in headerlets')